# 05 Clustering and payload side study (evidence for the decision)

Both are open decisions (report in results, appendix, or drop). This notebook produces the
numbers for that decision on all six experiments from the exported frames.

**Clustering.** Per address aggregates (mean, std, min, max of the model features, addresses
with at least `MIN_PACKETS` packets), standardised, k-means / BIRCH / bisecting k-means.
Two questions:
* all addresses, k = 3: do background, group 1 and group 2 form clusters?
* target addresses only, k = 2: does unsupervised clustering separate group 1 from group 2?
  This is the actual multi-group question and the one worth reporting.

Note: the clustering cells in the modelling notebooks keep `group_label_multi` and `group_id`
in the feature matrix (only `source` and `group_label` are dropped), so their ARI includes the
label itself. Here only model features are used.

**Payload.** For each target unit and byte position of the manufacturer data: is the byte
constant for that unit, does it differ between the two units of the same model (group 1 against
group 2), and is it the same in the training capture and the university capture? A byte that
satisfies all three identifies the unit across sessions. This is the upper bound described in
the design chapter, not a model input.

In [ ]:
import sys, pathlib, importlib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
sys.path.append(str(pathlib.Path.cwd()))  # the notebooks live in code/analysis
import analysis_utils as au
importlib.reload(au)

RESULTS = pathlib.Path('results')
FIGURES = RESULTS / 'figures'
FIGURES.mkdir(parents=True, exist_ok=True)

EXPS = au.available_experiments()
print('experiments with exported artifacts:', EXPS)
missing = [e for e in au.EXPERIMENTS if e not in EXPS]
if missing:
    print('NOT YET RERUN (no artifacts):', missing)

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans, Birch, BisectingKMeans
from sklearn.metrics import adjusted_rand_score, v_measure_score

MIN_PACKETS = 36
AGG = ['mean', 'std', 'min', 'max']

def aggregate(df, feats):
    counts = df['source'].value_counts()
    keep = counts[counts >= MIN_PACKETS].index
    d = df[df['source'].isin(keep)]
    X = d.groupby('source')[feats].agg(AGG)
    X.columns = ['_'.join(c) for c in X.columns]
    y = d.groupby('source')['group_label_multi'].agg(lambda s: s.value_counts().idxmax())
    dev_ = d.groupby('source')['labelled_device'].first()
    return X.fillna(0), y, dev_

def cluster(X, k):
    Xs = StandardScaler().fit_transform(X)
    return {'kmeans': KMeans(n_clusters=k, n_init=10, random_state=1).fit_predict(Xs),
            'birch': Birch(n_clusters=k).fit_predict(Xs),
            'bisecting_kmeans': BisectingKMeans(n_clusters=k, random_state=1).fit_predict(Xs)}

In [ ]:
rows = []
for e in EXPS:
    for which in ['train', 'eval']:
        df = au.load_frame(e, which)
        feats = au.feature_columns(df)
        X, y, dev_ = aggregate(df, feats)
        for scope, mask, k in [('all addresses, k=3', np.ones(len(y), bool), 3),
                               ('targets only, k=2', (y != 0).to_numpy(), 2)]:
            if mask.sum() < k + 1:
                continue
            for algo, lab in cluster(X[mask], k).items():
                rows.append(dict(exp=au.exp_name(e), capture=which, scope=scope, algorithm=algo,
                                 addresses=int(mask.sum()),
                                 targets=int((y[mask] != 0).sum()),
                                 ari=adjusted_rand_score(y[mask], lab),
                                 v_measure=v_measure_score(y[mask], lab)))
clus = pd.DataFrame(rows)
clus.to_csv(RESULTS / 'clustering.csv', index=False)
clus.pivot_table(index=['scope', 'algorithm', 'capture'], columns='exp', values='ari').round(3)

Reading guide: ARI 0 is chance, 1 is a perfect match. If the *targets only* ARI stays near 0
in the equal distance experiments, clustering cannot separate the two groups either, which is
a one line result for the thesis (appendix), not a section.

## Payload: unit specific constant bytes

In [ ]:
def byte_table(df):
    # one row per target packet with the manufacturer data split into byte columns
    t = df[(df['group_label_multi'] != 0) & df['data'].notna()][['labelled_device', 'group_label_multi', 'data']].copy()
    t['data'] = t['data'].astype(str)
    t = t[t['data'].str.fullmatch(r'[0-9a-fA-F]+') & (t['data'].str.len() % 2 == 0)]
    t['len'] = t['data'].str.len() // 2
    return t

DOMINANCE = 0.95   # a byte counts as constant if one value covers at least 95 % of the unit's packets
                   # (a few packets per unit are other frame variants or label noise)

def constant_bytes(t):
    # (device, group, length) -> {byte position: value} for positions that (almost) never change
    out = {}
    for (dev_, grp, ln), g in t.groupby(['labelled_device', 'group_label_multi', 'len']):
        if len(g) < 50:
            continue
        b = np.array([list(bytes.fromhex(s)) for s in g['data'].to_numpy()])
        const = {}
        for i in range(b.shape[1]):
            vals, cnt = np.unique(b[:, i], return_counts=True)
            if cnt.max() / len(b) >= DOMINANCE:
                const[i] = int(vals[cnt.argmax()])
        out[(dev_, int(grp), int(ln))] = const
    return out

rows = []
for e in EXPS:
    ct = constant_bytes(byte_table(au.load_frame(e, 'train', columns=['labelled_device', 'group_label_multi', 'data'])))
    ce = constant_bytes(byte_table(au.load_frame(e, 'eval', columns=['labelled_device', 'group_label_multi', 'data'])))
    for (dev_, grp, ln), const in ct.items():
        other = ct.get((dev_, 3 - grp, ln), {})
        ev_ = ce.get((dev_, grp, ln), {})
        for pos, val in const.items():
            if pos in other and other[pos] != val:      # differs between the two units of this model
                rows.append(dict(exp=au.exp_name(e), device=dev_, group=grp, payload_len=ln, byte=pos,
                                 value=f'{val:02x}', other_group_value=f'{other[pos]:02x}',
                                 same_in_university_capture=ev_.get(pos) == val))
payload = pd.DataFrame(rows, columns=['exp', 'device', 'group', 'payload_len', 'byte', 'value',
                                      'other_group_value', 'same_in_university_capture'])
payload.to_csv(RESULTS / 'payload_unit_bytes.csv', index=False)
payload if len(payload) else 'no unit specific constant byte found'

In [ ]:
# Summary: devices with at least one byte that identifies the unit across both sessions
if len(payload):
    (payload.groupby(['exp', 'device'])
     .agg(distinguishing_bytes=('byte', 'nunique'),
          stable_across_sessions=('same_in_university_capture', 'sum'))
     .reset_index())